#import book


In [1]:
import gdown
import pandas as pd
import numpy as np

# Define the file ID and download URL
train_file_id = '1Vpz3k6LI3aH7IO_yZMXAf_p2G-pM09a8'
train_download_url = f'https://drive.google.com/uc?id={train_file_id}'
train_output_file = 'train.csv'  # Adjusted file extension to .xlsx

# Download the file
gdown.download(train_download_url, train_output_file, quiet=False)

# Load the Excel file into a DataFrame
df_train = pd.read_csv(train_output_file)



Downloading...
From: https://drive.google.com/uc?id=1Vpz3k6LI3aH7IO_yZMXAf_p2G-pM09a8
To: /content/train.csv
100%|██████████| 12.9k/12.9k [00:00<00:00, 14.5MB/s]


In [2]:
import pandas as pd
import numpy as np

# Assuming df_train is your DataFrame
df_train['Genre'] = df_train['Genre'].fillna('Unknown')

In [3]:
df_train['index']=[i for i in range(len(df_train))]

In [4]:
df_train

,Book,Author(s),Original language,First published,Approximate sales in millions,Genre,index
0,A Tale of Two Cities,Charles Dickens,English,1859,200.0,Historical fiction,0
1,The Little Prince (Le Petit Prince),Antoine de Saint-Exupéry,French,1943,200.0,Novella,1
2,Harry Potter and the Philosopher's Stone,J. K. Rowling,English,1997,120.0,Fantasy,2
3,And Then There Were None,Agatha Christie,English,1939,100.0,Mystery,3
4,Dream of the Red Chamber (紅樓夢),Cao Xueqin,Chinese,1791,100.0,Family saga,4
...,...,...,...,...,...,...,...
169,The Goal,Eliyahu M. Goldratt,English,1984,10.0,Unknown,169
170,Fahrenheit 451,Ray Bradbury,English,1953,10.0,Unknown,170
171,Angela's Ashes,Frank McCourt,English,1996,10.0,Unknown,171
172,The Story of My Experiments with Truth (સત્યના...,Mohandas Karamchand Gandhi,Gujarati,1929,10.0,Unknown,172


In [5]:
df=df_train.copy()

In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 174 entries, 0 to 173
Data columns (total 7 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   Book                           174 non-null    object 
 1   Author(s)                      174 non-null    object 
 2   Original language              174 non-null    object 
 3   First published                174 non-null    int64  
 4   Approximate sales in millions  174 non-null    float64
 5   Genre                          174 non-null    object 
 6   index                          174 non-null    int64  
dtypes: float64(1), int64(2), object(4)
memory usage: 9.6+ KB


In [7]:
!pip install gradio

#eksp 3

In [8]:
import pandas as pd
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import gradio as gr

theme = gr.themes.Soft(
    primary_hue="violet",
    secondary_hue="violet",
    neutral_hue="violet",
).set(
    body_background_fill='*block_label_background_fill'
)

class User:
    def __init__(self, id_user, pass_user, nama_user, nomor_user, alamat_user):
        self.id_user = id_user
        self.pass_user = pass_user
        self.nama_user = nama_user
        self.nomor_user = nomor_user
        self.alamat_user = alamat_user

    def login(self, id_user, pass_user):
        return self.id_user == id_user and self.pass_user == pass_user

class Admin(User):
    def __init__(self, id_user, pass_user, nama_user, nomor_user, alamat_user):
        super().__init__(id_user, pass_user, nama_user, nomor_user, alamat_user)
        self.is_admin = True

class RegularUser(User):
    def __init__(self, id_user, pass_user, nama_user, nomor_user, alamat_user):
        super().__init__(id_user, pass_user, nama_user, nomor_user, alamat_user)
        self.is_admin = False

class BookManager:
    def __init__(self, df):
        self.df = df
        self.borrowed_books = {}  # Dictionary to store borrowed books: {user_id: [(book_id, days), ...]}
        self.users = []
        self.admins = []
        self.logged_in_user = None  # Track logged-in user

    def register_user(self, user_type, user_id, user_pass, user_nama, user_nomor, user_alamat):
        nomor_pattern = r'^(?:\+62|62|0)(\d{2,3})(\d{7,10})$'

        if user_nomor == "" or not re.match(nomor_pattern, user_nomor):
            return "Registrasi nomor telepon salah.", "", "", "", "", ""
        if user_id == "":
            return "ID tidak boleh kosong.", "", "", "", "", ""
        if user_pass == "":
            return "Password tidak boleh kosong.", "", "", "", "", ""
        if user_nama == "":
            return "Nama tidak boleh kosong.", "", "", "", "", ""
        if user_alamat == "":
            return "Alamat tidak boleh kosong.", "", "", "", "", ""

        if user_type == "Admin":
            pattern_admin = r'^\w+@mail\.unpad\.ac\.id$'
            if re.match(pattern_admin, user_id):
                admin = Admin(user_id, user_pass, user_nama, user_nomor, user_alamat)
                self.admins.append(admin)
                return "Registrasi admin berhasil.", "", "", "", "", ""
            else:
                return "Registrasi admin gagal: ID admin tidak valid.", "", "", "", "", ""
        else:
            user = RegularUser(user_id, user_pass, user_nama, user_nomor, user_alamat)
            self.users.append(user)
            return "Registrasi user berhasil.", "", "", "", "", ""

    def login_user(self, user_type, user_id, user_pass):
        if self.logged_in_user is not None:
          return "Login gagal: Anda harus logout terlebih dahulu sebelum login dengan akun lain.", "", ""
        else:
          if user_type == "User":
            for user in self.users:
              if user.login(user_id, user_pass):
                    self.logged_in_user = user
                    return f"Login berhasil sebagai user dengan ID {user_id}.", "", ""
            return "User login gagal. ID atau password tidak valid.", "", ""
          elif user_type == "Admin":
            pattern_admin = r'^\w+@mail\.unpad\.ac\.id$'
            if re.match(pattern_admin, user_id):
                for admin in self.admins:
                    if admin.login(user_id, user_pass):
                        self.logged_in_user = admin
                        return f"Login berhasil sebagai admin dengan ID {user_id}.", "", ""
                return "Admin login gagal. ID atau password tidak valid.", "", ""
            else:
                return "Admin login gagal: ID admin tidak valid.", "", ""

        return "Login gagal.", "", ""

    def logout_user(self):
        if self.logged_in_user is None:
          return "Tidak ada akun yang sedang login."
        elif self.logged_in_user is not None:
          self.logged_in_user = None
          return "Logout berhasil."

    def is_logged_in(self):
        return self.logged_in_user is not None

    def cari_buku(self, keyword):
        if not self.is_logged_in():
            return pd.DataFrame({"Message": ["Anda harus login terlebih dahulu."]})

        tfidf_vectorizer = TfidfVectorizer(stop_words='english')
        tfidf_matrix = tfidf_vectorizer.fit_transform(self.df['Book'] + " " + self.df['Author(s)'] + " " + self.df['Genre'] + " " + self.df['Original language'])
        keyword_matrix = tfidf_vectorizer.transform([keyword])
        cosine_similarities = cosine_similarity(keyword_matrix, tfidf_matrix).flatten()
        related_indices = cosine_similarities.argsort()[::-1]
        return self.df.iloc[related_indices[:10]]

    def pinjam_buku(self, user_id, book_index, days):
        if not self.is_logged_in():
            return "Anda harus login terlebih dahulu."

        if user_id == self.logged_in_user.id_user:
          if user_id in self.borrowed_books:
              self.borrowed_books[user_id].append((book_index, days))
          else:
              self.borrowed_books[user_id] = [(book_index, days)]
          return f"Buku dengan indeks {book_index} berhasil dipinjam selama {days} hari."
        else:
          return "Akses ditolak: Anda hanya bisa meminjam buku dengan akun yang sedang login."

    def kembalikan_buku(self, user_id, book_index):
        if not self.is_logged_in():
            return "Anda harus login terlebih dahulu."

        if user_id == self.logged_in_user.id_user:
          if user_id in self.borrowed_books:
              for i, (index, days) in enumerate(self.borrowed_books[user_id]):
                  if index == book_index:
                      self.borrowed_books[user_id].pop(i)
                      return f"Buku dengan judul {self.df['Book'].iloc[book_index]} telah berhasil dikembalikan."
          return "Buku tidak ditemukan dalam daftar peminjaman."
        else:
          return "Akses ditolak: Anda hanya bisa mengembalikan buku dengan akun yang sedang login."

    def tampilkan_buku_dipinjam(self, user_id):
        if not self.is_logged_in():
            return pd.DataFrame({"Message": ["Anda harus login terlebih dahulu."]})

        if user_id == self.logged_in_user.id_user:
          if user_id in self.borrowed_books and self.borrowed_books[user_id]:
              borrowed_book_indices = [index for index, days in self.borrowed_books[user_id]]
              borrowed_books_info = self.df.loc[borrowed_book_indices]
              return borrowed_books_info
          else:
              return pd.DataFrame({"Message": ["Tidak ada buku yang dipinjam."]})
        else:
          return pd.DataFrame({"Message": ["Akses ditolak: Anda hanya bisa melihat buku yang sedang dipinjam pada akun yang sedang login saja."]})

    def tampilkan_koleksi_buku(self):
        if not self.is_logged_in():
            return pd.DataFrame({"Message": ["Anda harus login terlebih dahulu."]})

        return self.df

    def is_admin_logged_in(self):
        return isinstance(self.logged_in_user, Admin)

    def tambah_buku(self, judul, penulis, genre, bahasa, tahun, penjualan):
        if not self.is_admin_logged_in():
          return "Akses ditolak: Hanya admin yang bisa menambah buku."
        else:
          try:
            # Konversi tahun ke integer
            tahun = int(tahun)
            # Ubah penjualan ke float
            penjualan = float(penjualan)

            # Tambahkan buku baru ke DataFrame
            new_book = {
              "Book": judul,
              "Author(s)": penulis,
              "Genre": genre,
              "Original language": bahasa,
              "First published": tahun,
              "Approximate sales in millions": penjualan
            }

            # Ambil indeks berikutnya setelah indeks terakhir
            next_index = self.df['index'].max() + 1 if not self.df.empty else 0

            # Tambahkan baris baru dengan menggunakan indeks berikutnya
            new_book['index'] = next_index
            self.df.loc[len(self.df)] = new_book

            return f"Buku '{judul}' berhasil ditambahkan."

          except ValueError:
            return "Gagal menambahkan buku: Tahun harus berupa bilangan bulat."

    def hapus_buku(self, book_index):
        if not self.is_admin_logged_in():
            return "Akses ditolak: Hanya admin yang bisa menghapus buku."
        else:
          if 0 <= book_index < len(self.df):
              book_title = self.df.at[book_index, "Book"]
              self.df = self.df.drop(book_index).reset_index(drop=True)
              return f"Buku '{book_title}' berhasil dihapus."
          else:
              return "Indeks buku tidak valid."

def main():
    def register(user_type, user_id, user_pass, user_nama, user_nomor, user_alamat):
        return book_manager.register_user(user_type, user_id, user_pass, user_nama, user_nomor, user_alamat)

    def login(user_type, user_id, user_pass):
      return book_manager.login_user(user_type, user_id, user_pass)

    def logout():
        return book_manager.logout_user()

    def search_book(keyword):
        return book_manager.cari_buku(keyword)

    def borrow_book(user_id, book_index, days):
        return book_manager.pinjam_buku(user_id, int(book_index), int(days))

    def return_book(user_id, book_index):
        return book_manager.kembalikan_buku(user_id, int(book_index))

    def show_borrowed_books(user_id):
        borrowed_books_info = book_manager.tampilkan_buku_dipinjam(user_id)
        return borrowed_books_info

    def show_all_books():
        return book_manager.tampilkan_koleksi_buku()

    def add_book(judul, penulis, genre, bahasa, tahun, penjualan):
        return book_manager.tambah_buku(judul, penulis, genre, bahasa, tahun, penjualan)

    def delete_book(book_index):
        return book_manager.hapus_buku(int(book_index))

     # HTML
    title_html = """
    <div style="text-align: center; margin-top: 20px;">
        <h1 style="color: #4C1D95; font-size: 40px;">
        📚 B I B L I O T H E C A 📚
        </h1>
    </div>
    """

    def create_interface():
        with gr.Blocks(theme=theme) as iface:
            gr.HTML(title_html)
            with gr.Tab("Registrasi"):
                gr.Markdown("## Registrasi Pengguna")
                with gr.Row():
                    with gr.Column():
                        user_type = gr.Radio(choices=["Admin", "User"], label="Tipe Pengguna (Admin/User)", value="u")
                        user_id = gr.Textbox(label="ID Pengguna")
                        user_pass = gr.Textbox(label="Password", type="password")
                        user_nama = gr.Textbox(label="Nama Pengguna")
                        user_nomor = gr.Textbox(label="Nomor Telepon")
                        user_alamat = gr.Textbox(label="Alamat")
                        registrasi_btn = gr.Button("Register")
                        registrasi_output = gr.Textbox(label="Hasil Registrasi")
                registrasi_btn.click(register, [user_type, user_id, user_pass, user_nama, user_nomor, user_alamat], [registrasi_output, user_id, user_pass, user_nama, user_nomor, user_alamat])

            with gr.Tab("Login/Logout"):
                gr.Markdown("## Login/Logout Pengguna")
                with gr.Row():
                    with gr.Column():
                        login_type = gr.Radio(choices=["Admin", "User"], label="Tipe Pengguna (Admin/User)", value="u")
                        login_id = gr.Textbox(label="ID Pengguna")
                        login_pass = gr.Textbox(label="Password", type="password")
                        login_btn = gr.Button("Login")
                        login_output = gr.Textbox(label="Hasil Login")
                    login_btn.click(login, [login_type, login_id, login_pass], [login_output, login_id, login_pass])
                with gr.Row():
                    logout_btn = gr.Button("Logout")
                    logout_output = gr.Textbox(label="Hasil Logout")
                    logout_btn.click(logout, None, logout_output)

            with gr.Tab("Cari Buku"):
                gr.Markdown("## Pencarian Buku")
                with gr.Row():
                    with gr.Column():
                        keyword = gr.Textbox(label="Kata Kunci Pencarian")
                        search_btn = gr.Button("Cari Buku")
                        search_output = gr.Dataframe(headers=["Judul", "Penulis", "Genre", "Bahasa", "Tahun", "Penjualan"])
                search_btn.click(search_book, [keyword], search_output)

            with gr.Tab("Pinjam Buku"):
                gr.Markdown("## Pinjam Buku")
                with gr.Row():
                    with gr.Column():
                        user_id = gr.Textbox(label="ID Pengguna")
                        book_index = gr.Number(label="Indeks Buku")
                        days = gr.Number(label="Jumlah Hari Peminjaman")
                        borrow_btn = gr.Button("Pinjam Buku")
                        borrow_output = gr.Textbox(label="Hasil Peminjaman")
                borrow_btn.click(borrow_book, [user_id, book_index, days], borrow_output)

            with gr.Tab("Kembalikan Buku"):
                gr.Markdown("## Kembalikan Buku")
                with gr.Row():
                    with gr.Column():
                        user_id = gr.Textbox(label="ID Pengguna")
                        book_index = gr.Number(label="Indeks Buku")
                        return_btn = gr.Button("Kembalikan Buku")
                        return_output = gr.Textbox(label="Hasil Pengembalian")
                return_btn.click(return_book, [user_id, book_index], return_output)

            with gr.Tab("Lihat Buku yang Dipinjam"):
                gr.Markdown("## Buku yang Dipinjam")
                with gr.Row():
                    with gr.Column():
                        user_id = gr.Textbox(label="ID Pengguna")
                        show_borrowed_btn = gr.Button("Lihat Buku Dipinjam")
                        show_borrowed_output = gr.Dataframe(headers=["Judul", "Penulis", "Genre", "Bahasa", "Tahun", "Penjualan"])
                show_borrowed_btn.click(show_borrowed_books, [user_id], show_borrowed_output)

            with gr.Tab("Lihat Koleksi Buku"):
                gr.Markdown("## Koleksi Buku")
                with gr.Row():
                    with gr.Column():
                        show_all_btn = gr.Button("Lihat Semua Buku")
                        show_all_output = gr.Dataframe(headers=["Judul", "Penulis", "Genre", "Bahasa", "Tahun", "Penjualan"])
                show_all_btn.click(show_all_books, None, show_all_output)

            with gr.Tab("Tambah Buku"):
                gr.Markdown("## Tambah Buku")
                with gr.Row():
                    with gr.Column():
                        judul = gr.Textbox(label="Judul Buku")
                        penulis = gr.Textbox(label="Penulis")
                        genre = gr.Textbox(label="Genre")
                        bahasa = gr.Textbox(label="Bahasa")
                        tahun = gr.Number(label="Tahun Terbit")
                        penjualan = gr.Number(label="Penjualan (dalam juta)")
                        add_book_btn = gr.Button("Tambah Buku")
                        add_book_output = gr.Textbox(label="Hasil Penambahan Buku")
                add_book_btn.click(add_book, [judul, penulis, genre, bahasa, tahun, penjualan], add_book_output)

            with gr.Tab("Hapus Buku"):
                gr.Markdown("## Hapus Buku")
                with gr.Row():
                    with gr.Column():
                        book_index = gr.Number(label="Indeks Buku")
                        delete_book_btn = gr.Button("Hapus Buku")
                        delete_book_output = gr.Textbox(label="Hasil Penghapusan Buku")
                delete_book_btn.click(delete_book, [book_index], delete_book_output)

        return iface

    book_manager = BookManager(df)

    # Launch Gradio interface
    iface = create_interface()
    iface.launch()

if __name__ == "__main__":
    main()

/tmp/ipykernel_1257/2732338698.py:250: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=theme) as iface:


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://f008a380e599bfee04.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
